<a href="https://colab.research.google.com/github/JARVIS-Materials-Design/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/ALIGNN_PhononDOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Rapid Prediction of Phonon Structure and Properties using an Atomistic Line Graph Neural Network (ALIGNN)

Refs:1) https://arxiv.org/abs/2207.12510,
2) https://www.nature.com/articles/s41524-021-00650-1

GitHub page: https://github.com/usnistgov/alignn

In [ ]:
!pip install alignn


1) Using script to predict PhononDOS

In [ ]:
from alignn.pretrained import  get_prediction,get_figshare_model
from jarvis.analysis.phonon.dos import PhononDos
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
from jarvis.core.graphs import Graph
import numpy as np 
import torch
#Ce: JVASP-14640
atoms=Atoms.from_dict(get_jid_data(jid='JVASP-14640',dataset='dft_3d')['atoms'])
model=get_figshare_model('jv_pdos_alignn')

device = "cpu"
# if torch.cuda.is_available():
#     device = torch.device("cuda")
model.to(device)
cutoff = 8
max_neighbors = 12
g, lg = Graph.atom_dgl_multigraph(
    atoms, cutoff=float(cutoff), max_neighbors=max_neighbors,
)
out_data = (
    model([g.to(device), lg.to(device)])
    .detach()
    .cpu()
    .numpy()
    .flatten()
    .tolist()
)
ph_dos = np.array(out_data)
ph_dos[ph_dos<0]=0
phonon_freq_cm=np.linspace(-300,1000,66)
ph_dos = ph_dos[phonon_freq_cm>0]
phonon_freq_cm = phonon_freq_cm[phonon_freq_cm>0]


f=open('ph_dos_data.csv','w')
f.write('Frequency_cm,DOS\n')
for i,j in zip(phonon_freq_cm,ph_dos):
    line = str(i)+','+str(j)+'\n'
    f.write(line)
f.close()

ph=PhononDos(phonon_dos=ph_dos,phonon_freq_cm=phonon_freq_cm)
debye_temp = ph.debye_temperature(atoms)
print ('debye_temp',debye_temp)

In [ ]:
print(atoms)

In [ ]:
!pip uninstall -y matplotlib
!pip install matplotlib==3.1.3

In [ ]:

import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(phonon_freq_cm,ph_dos)
plt.xlabel('Freq(cm$^-1$)')
plt.ylabel('DOS')


2) Using command line

In [ ]:
atoms.write_poscar('POSCAR')

In [ ]:
!ls

We will predict PhononDOS given this POSCAR file. The pretrained.py automatically comes with alignn installation.

In [ ]:
!pretrained.py --model_name jv_pdos_alignn --file_format poscar --file_path POSCAR